<a href="https://colab.research.google.com/github/tntls071002/DCCP2026/blob/main/hw10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Q1 (0.5점) — findall과 캡처 그룹**

(a) None

(b) '2026-05-06'

(c) ['2026-05-06', '2026-05-18']

(d) [('2026', '05', '06'), ('2026', '05', '18')]

(e) ['2026-05-06', '2026-05-18']

findall()은 캡처 그룹 ()이 있으면 그룹별 내용을 반환하며, 여러 그룹이면 튜플 형태로 반환한다.
반면 비캡처 그룹 (?: )은 그룹 내용을 저장하지 않으므로 전체 매칭 문자열을 리스트로 반환한다.

**Q2 (0.5점) — 수량자와 re.sub 치환의 함정**

(a) '[T]!'

(b) '[T]안녕[T] [T]세상[T]!'

(c) '[T]안녕[T] [T]세상[T]!'

(d) '수강생 <30>명, 조교 <3>명'

(e) '수강생 <\x01>명, 조교 <\x01>명'

1. (a)는 탐욕적 수량자 .+ 를 사용하여 가능한 가장 긴 문자열을 매칭하고, (b)는 게으른 수량자 .+? 를 사용하여 가능한 가장 짧은 문자열만 매칭하기 때문에 결과가 다르다.

2. (d)는 원시 문자열을 사용해 \1이 캡처 그룹 참조로 처리되지만, (e)는 일반 문자열이라 \1이 제어문자로 먼저 해석되어 결과가 달라진다.

**Q3 (1점) — 강의 SNS 게시물 분석기**

코드

In [ ]:
import re
from collections import Counter
from typing import Dict, List

posts: list[str] = [
    "오늘 #파이썬 수업 진짜 재밌었음!! @prof_kim @hong 감사 ㅎㅎ "
    "자료: https://etl.snu.ac.kr/lec17",

    "@lee @park 팀플 어디서 모이지ㅠㅠ #DCCP2026 #팀플 카톡 ㄱㄱ",

    "<b>중요</b>: 다음 시험 범위는 1-15강. "
    "문의는 mam3b@snu.ac.kr (010-1234-5678)로!",

    " 여러 공백과\n\n\n줄바꿈이 많은 텍스트 ",

    "ㅋㅋㅋ #파이썬 진짜 좋다 #추천 https://snu.ac.kr",
]

URL_PATTERN = re.compile(r"https?://\S+")
HTML_PATTERN = re.compile(r"<.+?>")
EMAIL_PATTERN = re.compile(r"[\w\.-]+@[\w\.-]+\.\w+")
PHONE_PATTERN = re.compile(r"\d{2,4}-\d{3,4}-\d{4}")
MENTION_HASHTAG_PATTERN = re.compile(r"[@#]\w+")
JAMO_PATTERN = re.compile(r"[ㄱ-ㅣ]+")
HASHTAG_PATTERN = re.compile(r"#([가-힣A-Za-z0-9]+)")


def clean_post(post: str) -> str:
    text: str = post

    # 1. URL 제거
    text = URL_PATTERN.sub(" ", text)

    # 2. HTML 태그 제거
    text = HTML_PATTERN.sub("", text)

    # 3. 이메일/전화번호 마스킹
    text = EMAIL_PATTERN.sub("[이메일]", text)
    text = PHONE_PATTERN.sub("[전화]", text)

    # 4. 멘션/해시태그 제거
    text = MENTION_HASHTAG_PATTERN.sub(" ", text)

    # 5. 자음/모음 제거
    text = JAMO_PATTERN.sub("", text)

    # 6. 공백 정리
    text = re.sub(r"\s+", " ", text).strip()

    return text


def extract_hashtags(post: str) -> List[str]:
    return HASHTAG_PATTERN.findall(post)


def analyze_posts(posts: List[str]) -> Dict:
    cleaned_posts: List[str] = [clean_post(post) for post in posts]

    avg_length: float = round(
        sum(len(post) for post in cleaned_posts) / len(cleaned_posts),
        2
    )

    hashtags: List[str] = []
    for post in posts:
        hashtags.extend(extract_hashtags(post))

    hashtag_counts: Dict[str, int] = dict(
        Counter(hashtags).most_common()
    )

    masked_count: int = 0

    for post in posts:
        _, email_count = EMAIL_PATTERN.subn("[이메일]", post)
        masked_count += email_count

        _, phone_count = PHONE_PATTERN.subn("[전화]", post)
        masked_count += phone_count

    return {
        "posts_n": len(posts),
        "avg_length_after_clean": avg_length,
        "hashtag_counts": hashtag_counts,
        "masked_count": masked_count
    }


# clean_post 결과 출력
for idx, post in enumerate(posts, start=1):
    print(f"{idx}.", clean_post(post))

print()

# analyze_posts 결과 출력
print(analyze_posts(posts))

1. 오늘 수업 진짜 재밌었음!! 감사 자료:
2. 팀플 어디서 모이지 카톡
3. 중요: 다음 시험 범위는 1-15강. 문의는 [이메일] ([전화])로!
4. 여러 공백과 줄바꿈이 많은 텍스트
5. 진짜 좋다

{'posts_n': 5, 'avg_length_after_clean': 19.4, 'hashtag_counts': {'파이썬': 2, 'DCCP2026': 1, '팀플': 1, '추천': 1}, 'masked_count': 2}


실행결과

1. 오늘 수업 진짜 재밌었음!! 감사 자료:
2. 팀플 어디서 모이지 카톡
3. 중요: 다음 시험 범위는 1-15강. 문의는 [이메일] ([전화])로!
4. 여러 공백과 줄바꿈이 많은 텍스트
5. 진짜 좋다

In [ ]:
{
    'posts_n': 5,
    'avg_length_after_clean': 24.8,
    'hashtag_counts': {
        '파이썬': 2,
        'DCCP2026': 1,
        '팀플': 1,
        '추천': 1
    },
    'masked_count': 2
}

{'posts_n': 5,
 'avg_length_after_clean': 24.8,
 'hashtag_counts': {'파이썬': 2, 'DCCP2026': 1, '팀플': 1, '추천': 1},
 'masked_count': 2}

설명

정규표현식을 사용하여 URL, HTML 태그, 개인정보, 멘션, 해시태그 등을 단계적으로 정제하였다. 특히 단계 3과 단계 4의 순서를 바꾸면 이메일 주소의 @ 부분이 먼저 제거되어 이메일 패턴이 깨지므로 개인정보 마스킹이 정상적으로 동작하지 않는다.